In [2]:
import pandas as pd
pd.options.display.float_format = '{:.2e}'.format
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt

from Example1_DNA_dsb_repair.utils.utils import *

from InDelsTopo import Filtration

with open('Example1_DNA_dsb_repair/data/preprocessed/Human_experiments_average.pickle', 'rb') as handle:
    Average_Experiments = pkl.load(handle)

In [3]:
sample_n=50
min_threshold=1e-5
max_dim=10

#x-coordinates for computing Euler Curves
xs=np.power(10,np.linspace(-7,0,num=sample_n))

#Create matrix to save Euler Curves, and lists to sava labels and complexes
X=np.zeros((len(Average_Experiments),sample_n))
labels=[]
Filtrations=[]

#Get Words
Experiment=Average_Experiments[0]
Table=Experiment['table'].sort_values('avg_freq',ascending=True)

#Filter by threshold for average frequency
Table=Table[Table.avg_freq>=min_threshold]

Words=list(Table.sequence)
Freqs=np.array(Table.avg_freq)

#normalize Freqs
Freqs=Freqs/sum(Freqs)

Words=['a','b','ab','abc','aba','bbc','bba']

In [20]:
import re
from collections import defaultdict

def replacement_block(s1, s2):
    s1,symbols=replace_brackets(s1,'x')
    if 'x' in s2:
        print('ERROR',s1,s2)
    seen = None
    for i, (a, b) in enumerate(zip(s1, s2)):
        if a != b:
            if seen is not None:
                return False, None, None
            seen = i
    if seen is None:
        return False, None,None
    symbols=symbols+[s1[seen], s2[seen]]
    symbols.remove('x') if 'x' in symbols else None
    symbols=sorted(list(set(symbols)))
    return (False, None, None) if seen is False else True, s1[:seen]+'['+''.join(symbols)+']'+s1[seen+1:], symbols


def replacement_block(s1, s2,new_symbols):
    # s1,symbols=replace_brackets(s1,'x')
    # if 'x' in s2:
    #     print('ERROR',s1,s2)
    symbols=[]
    seen = None
    for i, (a, b) in enumerate(zip(s1, s2)):
        if a != b:
            if seen is not None:
                return False, None, None
            seen = i
    if seen is None:
        return False, None,None
    symbols=symbols+[s1[seen], s2[seen]]
    symbols.remove('x') if 'x' in symbols else None
    symbols=sorted(list(set(symbols)))
    return (False, None, None) if seen is False else True, s1[:seen]+[symbols]+s1[seen+1:], symbols



def replace_brackets(s, sym='x'):
    m = re.search(r'\[(.*?)\]', s)
    return (re.sub(r'\[.*?\]', sym, s), list(m.group(1))) if m else (s, [])

def group_by_length(words):
    d = defaultdict(list)
    for w in words:
        d[len(w)].append(w)
    return dict(d)

In [21]:
replacement_block(['a','b','c'],['a','c','c'],{})

(True, ['a', ['b', 'c'], 'c'], ['b', 'c'])

In [6]:
Words_dict=group_by_length(Words)
if 0 in Words_dict:
    del Words_dict[0]

In [8]:
Blocks_dim

{0: {1: ['a', 'b'], 2: ['ab'], 3: ['abc', 'aba', 'bbc', 'bba']},
 1: {1: {'[ab]'}, 3: {'[ab]ba', '[ab]bc', 'ab[ac]', 'bb[ac]'}}}

In [7]:
Blocks_dim={0:Words_dict}
New_Blocks_dict={}
for dim in range(3):
    for ell in Blocks_dim[dim]:
        for w1 in Blocks_dim[dim][ell]:
            for w2 in Words_dict[ell]:
                is_block,block,symbols=replacement_block(w1,w2)
                if is_block:
                    new_dim=len(symbols)-1
                    if not(new_dim in Blocks_dim):
                        Blocks_dim[new_dim]={}
                    Blocks_dim[new_dim].setdefault(ell, set()).add(block)
        
    

KeyError: 2

In [147]:
replacements={}
max_replacement=0
New_words=[]
for dim in Blocks_dim:
    for ell in Blocks_dim[dim]:
        for word in Blocks_dim[dim][ell]:
            new_word, symbols=replace_brackets(word,'x')
            symbols=''.join(sorted(list(set(symbols))))
            if symbols in replacements:
                y=replacements[symbols]
            else:
                max_replacement+=1
                y=max_replacement
                replacements[symbols]=y
            new_word='*'.join(new_word)
            new_word=new_word.replace('x','x_'+str(y))
            New_words.append(new_word)
                

In [148]:
from InDelsTopo import Complex
K=Complex()
K.compute_d_skeleton(New_words)

Product symbol set to *


In [1]:
%matplotlib widget
K.get_graph()

NameError: name 'K' is not defined

In [150]:
New_words

['a',
 'b',
 'a*b',
 'a*b*c',
 'a*b*a',
 'b*b*c',
 'b*b*a',
 'x_2',
 'b*b*x_3',
 'a*b*x_3',
 'x_2*b*c',
 'x_2*b*a']